# Value-Based Reinforcement Learning · 실습


## 학습 목표

<!-- exercise-guidance:reinforcement-learning/week-02/objective -->
작은 문제를 Table로 풀어 두면, 복잡한 함수 근사 모델을 비교할 기준을 만들 수 있습니다.
<!-- /exercise-guidance:reinforcement-learning/week-02/objective -->


## 실행 조건

<!-- exercise-guidance:reinforcement-learning/week-02/execution -->
기준 Queue 예제는 NumPy로 실행합니다. 학습에 사용하지 않은 seed=100…104에서 정책을 비교합니다.
<!-- /exercise-guidance:reinforcement-learning/week-02/execution -->


## 적용 사례

<!-- exercise-guidance:reinforcement-learning/week-02/case-studies -->
### 장비·통신 자원의 처리량과 비용

합성 대기열에 매 시점 평균 2건이 도착합니다. 서비스 용량을 높이면 대기는 줄지만 에너지 비용이 늘어납니다.

**Observation·State:** 대기 수 q=0…20. 관측 q/20.

**행동:** 처리 용량 0·2·4 중 선택.

**Reward·Constraint:** −(다음 대기 수 + 0.2×용량² + 10×유실 수). 100 step에서 시간 제한.

**비교 기준:** 항상 용량 2, 대기 수에 따른 Threshold policy.

**확인할 지표:** 평가 Return, 평균 대기 수, 평균 용량², 유실 수.

### 이동로봇의 경로와 안전

격자 경로 또는 연속 속도 제어 중 한 가지를 선택하는 적용 설계 사례입니다.

**Observation·State:** 위치·목표 상대 위치·근접 장애물 관측.

**행동:** 격자 방향 선택 또는 제한된 Linear Velocity·Angular Velocity.

**Reward·Constraint:** 도착·충돌·이동 비용을 분리. 충돌과 시간 초과를 구분.

**비교 기준:** 최단 경로, 규칙 기반 회피.

**확인할 지표:** 성공률, 충돌률, 이동 거리, 시간. 미관측 지도에서 평가.
<!-- /exercise-guidance:reinforcement-learning/week-02/case-studies -->


## 작은 State Space에서 Q-learning

<!-- exercise-guidance:reinforcement-learning/week-02/activity-1 -->
### 시작 조건

Queue의 q=0…20, 행동 3개를 Q-table로 학습합니다. γ=.95, α=.1, ε=.2, 50 Episode, Episode당 100 step입니다.

### 진행 방법

1. 제공된 학습 루프를 실행하고 q=0,5,10,20의 Q값·선택 행동을 표로 적으세요.
2. 평가에서는 ε=0으로 두고 규칙 정책과 같은 seed에서 비교하세요.
3. 큰 대기 수에서 낮은 처리량을 고른다면 방문 횟수와 보상 구조를 먼저 확인하세요.

### 남길 결과

- 상태별 Q 표 일부, 정책 비교 표, 학습 Episode별 Return 그림.
<!-- /exercise-guidance:reinforcement-learning/week-02/activity-1 -->


In [ ]:
"""Synthetic service queue. An action chooses capacity, including its energy cost."""
import numpy as np

class Queue:
    def reset(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.q, self.t = 0, 0
        return self.q

    def step(self, action):
        capacity = (0, 2, 4)[int(action)]
        arriving = int(self.rng.poisson(2))
        raw = max(0, self.q - capacity) + arriving
        dropped = max(0, raw - 20)
        self.q = min(20, raw)
        self.t += 1
        reward = -(self.q + .2 * capacity**2 + 10 * dropped)
        return self.q, reward, False, self.t >= 100, {'queue': self.q, 'capacity': capacity, 'dropped': dropped}

def evaluate(policy, seeds=range(100, 105)):
    rows = []
    for seed in seeds:
        env = Queue(); q = env.reset(seed)
        total = waiting = energy = dropped = 0
        while True:
            q, r, terminated, truncated, info = env.step(policy(q))
            total += r; waiting += q; energy += info['capacity']**2; dropped += info['dropped']
            if terminated or truncated: break
        rows.append([total, waiting / env.t, energy / env.t, dropped])
    return np.array(rows)

policies = {'constant': lambda q: 1, 'threshold': lambda q: 0 if q == 0 else (1 if q < 5 else 2)}
for name, policy in policies.items():
    print(name, evaluate(policy).mean(axis=0))

Q=np.zeros((21,3));visits=np.zeros((21,3),dtype=int);history=[]
rng=np.random.default_rng(7)
for episode in range(50):
    env=Queue();q=env.reset(episode);total=0
    for k in range(100):
        a=int(rng.integers(3)) if rng.random()<.2 else int(Q[q].argmax())
        nq,r,te,tr,_=env.step(a)
        target=r+.95*(not te)*Q[nq].max()
        Q[q,a]+=.1*(target-Q[q,a]);visits[q,a]+=1;q=nq;total+=r
        if te or tr:break
    history.append(total)
print(evaluate(lambda q:int(Q[q].argmax())).mean(0))


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-02/activity-1-record -->
**q:**

**Q(a=0):**

**Q(a=1):**

**Q(a=2):**

**선택 행동:**

**방문 수:**

### 확인할 점

- 행동 선택과 평가 시 탐색을 구분했는가?
- 시간 제한에서 Bootstrap을 유지하고 평가 seed를 학습과 분리했는가?
<!-- /exercise-guidance:reinforcement-learning/week-02/activity-1-record -->


## Discount factor와 의사결정

<!-- exercise-guidance:reinforcement-learning/week-02/activity-2 -->
### 시작 조건

γ=.5와 .95를 비교합니다. 나머지 학습 조건·총 Transition 수는 고정합니다.

### 진행 방법

1. 기준 코드를 함수로 묶고 γ만 바꾸어 실행하세요.
2. 동일 평가군에서 대기·용량²·유실을 비교하세요.
3. 특정 q에서 선택 행동이 바뀌었다면 단기 비용과 미래 대기의 관점으로 설명하세요.

### 남길 결과

- 두 설정 비교 표와 q별 선택 행동 그림.
<!-- /exercise-guidance:reinforcement-learning/week-02/activity-2 -->


In [ ]:
# TODO: 학습 루프를 train_q(gamma)로 묶어 비교하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-02/activity-2-record -->
**γ:**

**평가 Return:**

**평균 대기:**

**용량²:**

**유실:**

**정책이 달라진 q:**

### 확인할 점

- 학습량까지 함께 바꾸지 않았는가?
- 우연한 한 Episode보다 공통 평가군으로 비교했는가?
<!-- /exercise-guidance:reinforcement-learning/week-02/activity-2-record -->


## 본인의 연구 문제로 옮기기

<!-- exercise-guidance:reinforcement-learning/week-02/activity-3 -->
### 시작 조건

앞의 실험을 바탕으로 Queue·Motor 또는 본인의 연구 문제 하나를 선택해, 실제 적용을 위한 실험을 설계합니다.

### 진행 방법

1. State·Observation·Action·Reward·종료·시간 제한을 표 한 장으로 정의하세요. 각 변수의 단위와 실제로 얻을 수 있는 정보를 적습니다.
2. 예상 Transition 3개를 수치로 적어, 좋은 행동과 나쁜 행동의 보상이 의도에 맞는지 확인하세요.
3. 규칙 기반 비교 기준 한 개, 평가 조건 한 개, 가장 우려되는 실패 사례 한 개를 정하세요. 필요한 실측 자료나 Simulator도 적습니다.

### 남길 결과

- MDP 설계 표, Transition 예 3개, 비교 실험 계획.
- 실제 연구 자료가 없으면 적용 사례의 합성 조건으로 설계합니다.
<!-- /exercise-guidance:reinforcement-learning/week-02/activity-3 -->


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-02/activity-3-record -->
**항목:**

**구체적 정의·단위:**

**관측 가능 여부·확인 방법:**

### 확인할 점

- 미래 정보나 평가 Label을 관측에 넣지 않았는가?
- 학습 Return만으로 연구 기여를 판단하지 않고, 해당 도메인의 지표를 정했는가?
<!-- /exercise-guidance:reinforcement-learning/week-02/activity-3-record -->
